# A1 - Test + analysis in one Run all

Chạy notebook này trong **phiên Colab mới** sau khi đã có đủ 15 run trên Drive.
Không cần chạy lại notebook training. Không có lệnh train, backward hay optimizer step.

- Nạp đúng commit từ `run_ledger.json`, đúng effective config và `best.pt` của từng run.
- Tự chọn GPU nếu có, nếu không dùng CPU cho validation/test/analysis.
- Xuất learning curves, predictions, confusion matrices, per-class metrics và ba gallery: đúng, đúng nhưng khó, sai.
- Kiểm tra validation replay trước khi đọc test; test đủ 10.000 ảnh cho cả 15 checkpoint.
- Đo inference và lưu timing samples/parameter count khi có GPU. Nếu không có GPU, ghi **skipped_gpu_unavailable**, không ghi timing CPU thành timing GPU.
- Lưu từng bước lên Drive; khi phiên bị ngắt, chạy lại notebook sẽ bỏ qua phần đã hoàn tất và kiểm tra được hash.
- Cuối cùng xuất bảng từng run, mean ± sample SD (n=3), biểu đồ so sánh và một ZIP tải về.

Mặc định đọc `MyDrive/dlbench_a1_main`; chỉ đổi đường dẫn ở cell 1 nếu nhóm đã đổi tên folder.
Chạy **Runtime → Run all** và chấp nhận mount Drive khi Colab yêu cầu. Không dùng số test để tune lại các model trong bộ so sánh này.


In [ ]:
from pathlib import Path
import os

ARTIFACT_ROOT = Path('/content/drive/MyDrive/dlbench_a1_main')
OUTPUT_ROOT = ARTIFACT_ROOT / 'postprocess_v1'
CHECKOUT = Path('/content/deep-learning-benchmark-suite')
REPO_URL = 'https://github.com/TuanKhai1210/Deep-Learning-Benchmark-Suite.git'
DEVICE = 'auto'          # 'auto', 'cpu', or 'cuda'
RUN_GPU_BENCHMARK = True # Automatically skipped on CPU.
AUTO_DOWNLOAD_ZIP = True

os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
os.environ['MPLBACKEND'] = 'Agg'


## 1. Mount Drive and restore the training revision

Nguồn được khóa theo ledger; không checkout main mới nhất để thay model đã train.
Cell cài các dependencies khai báo trong repository; môi trường thực tế sẽ được ghi trong session.json.


In [ ]:
import json
import subprocess
import sys
from google.colab import drive

drive.mount('/content/drive')
ledger_path = ARTIFACT_ROOT / 'run_ledger.json'
if not ledger_path.is_file():
    raise FileNotFoundError(f'Không thấy {ledger_path}. Kiểm tra ARTIFACT_ROOT.')
training_ledger = json.loads(ledger_path.read_text())
pinned_commit = training_ledger['identity']['git_commit']
if len(pinned_commit) != 40 or any(c not in '0123456789abcdef' for c in pinned_commit.lower()):
    raise ValueError('Invalid ledger commit')
if not CHECKOUT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(CHECKOUT)], check=True)
if not (CHECKOUT / '.git').is_dir():
    raise RuntimeError(f'Not a Git checkout: {CHECKOUT}')

def source_git(*args):
    return subprocess.run(['git', *args], cwd=CHECKOUT, check=True,
                          capture_output=True, text=True).stdout.strip()

if source_git('status', '--porcelain'):
    raise RuntimeError('Source checkout has local changes; preserve them and use a fresh runtime.')
try:
    source_git('cat-file', '-e', pinned_commit + '^{commit}')
except subprocess.CalledProcessError:
    subprocess.run(['git', 'fetch', 'origin', pinned_commit], cwd=CHECKOUT, check=True)
if source_git('rev-parse', 'HEAD') != pinned_commit and any(
    name == 'dlbench' or name.startswith('dlbench.') for name in sys.modules):
    raise RuntimeError('Old dlbench is already imported. Restart the runtime before changing revision.')
subprocess.run(['git', 'checkout', '--detach', pinned_commit], cwd=CHECKOUT, check=True)
os.chdir(CHECKOUT)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[ml]'], check=True)
sys.path.insert(0, str(CHECKOUT / 'src'))
if source_git('rev-parse', 'HEAD') != pinned_commit or source_git('status', '--porcelain'):
    raise RuntimeError('Source revision/clean-state gate failed')
print('Pinned source:', pinned_commit)


## 2. Postprocessing helpers

Cell này tự chứa toàn bộ mã điều phối, không cần upload thêm file Python.
Evaluator, plotting cơ bản và benchmark được dùng lại từ repository đã khóa commit.
Raw images chỉ được dùng để vẽ gallery theo đúng sample IDs; không sửa train/test split.

Validation replay được đối chiếu với metrics gốc. Khi dùng CPU, cho phép sai khác số học nhỏ:
accuracy tối đa một mẫu trên 10.000, macro-F1 tối đa 0,0002, loss tối đa 0,00001
(kèm relative tolerance 1e-5). Mọi delta/tolerance được lưu; vượt ngưỡng sẽ dừng trước test.
GPU dùng kiểm tra chặt hơn theo helper gốc. Đây là kiểm tra numerical replay, không phải tiêu chí chọn model.


In [ ]:
"""A1 postprocessing only. Embedded verbatim in the portable Colab notebook."""

from pathlib import Path
from copy import deepcopy
from dataclasses import asdict
from datetime import datetime, timezone
from statistics import mean, stdev
from uuid import uuid4
import csv
import hashlib
import json
import math
import os
import zipfile

VERSION = 'a1-postprocess-20261024-v1'
MODELS = ('linear', 'mlp', 'cnn', 'rnn', 'transformer')
SEEDS = (69420, 67, 69)
CLASS_NAMES = ('T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot')


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def save_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, ensure_ascii=False,
                                    allow_nan=False) + '\n', encoding='utf-8')


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def json_digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True,
        separators=(',', ':'), allow_nan=False).encode()).hexdigest()


def completed_stage(destination, identity, action):
    """Publish a stage only after all files finish; validate cached bytes on reuse."""
    destination = Path(destination)
    marker = destination / '_complete.json'
    if destination.exists():
        if not marker.is_file():
            raise RuntimeError(f'Unverified output folder; preserve and inspect: {destination}')
        saved = read_json(marker)
        if saved['identity'] != identity:
            raise RuntimeError(f'Output/input identity changed: {destination}')
        for relative, digest in saved['files'].items():
            path = (destination / relative).resolve()
            if not path.is_relative_to(destination.resolve()) or not path.is_file() or sha256(path) != digest:
                raise RuntimeError(f'Cached artifact missing/changed: {path}')
        print('  SKIP verified:', destination.name, flush=True)
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_name('.' + destination.name + '.partial-' + uuid4().hex[:10])
    # Each action creates its own new directory; repository helpers refuse overwrite.
    action(temporary)
    files = {p.relative_to(temporary).as_posix(): sha256(p)
             for p in sorted(temporary.rglob('*')) if p.is_file()}
    if not files:
        raise RuntimeError('Stage produced no artifacts')
    save_json(temporary / '_complete.json', {'identity': identity, 'files': files})
    temporary.rename(destination)
    return destination


def inspect_inputs(artifact_root):
    root = Path(artifact_root)
    ledger = read_json(root / 'run_ledger.json')
    expected = {f'{model}:{seed}' for model in MODELS for seed in SEEDS}
    if set(ledger['completed']) != expected:
        raise RuntimeError('Expected exactly the 15 agreed completed pairs in run_ledger.json')
    if set(ledger['identity']['models']) != set(MODELS) or set(ledger['identity']['seeds']) != set(SEEDS):
        raise RuntimeError('Ledger model/seed plan differs')
    items, signatures = [], set()
    for model in MODELS:
        for seed in SEEDS:
            run_name = ledger['completed'][f'{model}:{seed}']
            if Path(run_name).name != run_name:
                raise RuntimeError('Run ID must be one folder name')
            run = root / 'main' / run_name
            config, metadata, saved = [read_json(run / f'{name}.json')
                for name in ('config', 'metadata', 'metrics')]
            if (metadata['model_name'], metadata['run_seed'], metadata['run_mode'], metadata['git_dirty']) != (model, seed, 'main', False):
                raise RuntimeError(f'Unexpected run metadata: {run_name}')
            if metadata['git_revision'] != ledger['identity']['git_commit']:
                raise RuntimeError(f'Code revision differs: {run_name}')
            if metadata['split']['sha256'] != ledger['identity']['split_hash'] or metadata['normalization']['sha256'] != ledger['identity']['statistics_hash']:
                raise RuntimeError(f'Data provenance differs: {run_name}')
            if config['model']['name'] != model or config['run']['seed'] != seed or saved['eval_split'] != 'validation':
                raise RuntimeError(f'Config/metrics identity differs: {run_name}')
            if config['protocol']['status'] != 'frozen' or config['data']['split_seed'] != 36:
                raise RuntimeError('Expected frozen protocol and split seed 36')
            inputs = ['config.json', 'metadata.json', 'metrics.json', 'history.csv', 'best.pt']
            for record in metadata['sources'].values():
                snapshot = (run / record['snapshot']).resolve()
                if not snapshot.is_relative_to(run.resolve()) or sha256(snapshot) != record['sha256']:
                    raise RuntimeError(f'Snapshot hash mismatch: {snapshot}')
                inputs.append(record['snapshot'])
            hashes = {name: sha256(run / name) for name in inputs}
            signature = {k: config[k] for k in ('protocol', 'data', 'preprocessing', 'evaluation', 'checkpoint', 'budget', 'timing')}
            signatures.add(json_digest(signature))
            items.append({'model': model, 'seed': seed, 'run': run, 'config': config,
                          'metadata': metadata, 'saved': saved, 'hashes': hashes})
    if len(signatures) != 1:
        raise RuntimeError('Main runs do not share the same evaluation protocol')
    return ledger, items


def check_validation_replay(measured, saved, device):
    # CPU and CUDA can differ near numerical ties. CPU allowance is declared,
    # not a tuning rule: at most one accuracy count out of 10,000, F1 2e-4.
    tolerances = ({'loss': 1e-5, 'accuracy': 1e-4 + 1e-12, 'macro_f1': 2e-4}
                  if device == 'cpu' else {'loss': 1e-6, 'accuracy': 1e-6, 'macro_f1': 1e-6})
    deltas = {}
    for metric, tolerance in tolerances.items():
        actual, expected = measured[metric], saved['val_' + metric]
        deltas[metric] = actual - expected
        if not math.isfinite(actual) or not math.isclose(actual, expected, rel_tol=1e-5, abs_tol=tolerance):
            raise RuntimeError(f'Validation replay differs for {metric}: {actual} vs {expected}. Stop before test.')
    return {'deltas': deltas, 'absolute_tolerances': tolerances,
            'relative_tolerance': 1e-5, 'status': 'passed'}


def prediction_extras(predictions, destination, raw_datasets, title):
    import numpy as np
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    payload = read_json(destination / 'errors.json')
    matrix = np.asarray(payload['confusion_counts'], dtype=float)
    support, predicted = matrix.sum(axis=1), matrix.sum(axis=0)
    tp = matrix.diagonal()
    precision = np.divide(tp, predicted, out=np.zeros(10), where=predicted != 0)
    recall = np.divide(tp, support, out=np.zeros(10), where=support != 0)
    f1 = np.divide(2 * precision * recall, precision + recall,
                   out=np.zeros(10), where=precision + recall != 0)
    with (destination / 'per_class.csv').open('w', newline='', encoding='utf-8') as stream:
        writer = csv.writer(stream)
        writer.writerow(['class_id', 'class_name', 'support', 'precision', 'recall', 'f1'])
        for i in range(10):
            writer.writerow([i, CLASS_NAMES[i], int(support[i]), precision[i], recall[i], f1[i]])
    correct, wrong = [], []
    for sample_id, target, predicted_label, probabilities in zip(
        predictions.sample_ids, predictions.targets, predictions.predicted_labels, predictions.probabilities):
        row = (sample_id, target, predicted_label, probabilities[predicted_label])
        (correct if target == predicted_label else wrong).append(row)
    selections = {
        'correct': sorted(correct, key=lambda x: x[0])[:12],
        'difficult_correct': sorted(correct, key=lambda x: (x[3], x[0]))[:12],
        'errors': sorted(wrong, key=lambda x: (-x[3], x[0]))[:12],
    }
    save_json(destination / 'gallery_selection.json', {
        'policy': 'correct: first IDs; difficult_correct: lowest confidence; errors: highest confidence; at most 12 each',
        'samples': selections})
    for kind, selected in selections.items():
        figure = Figure(figsize=(11, 8), constrained_layout=True)
        FigureCanvasAgg(figure)
        axes = figure.subplots(3, 4).ravel()
        for axis in axes:
            axis.axis('off')
        for axis, (sample_id, target, predicted_label, confidence) in zip(axes, selected):
            dataset_name, namespace, index = sample_id.split(':')
            if dataset_name != 'fashion_mnist' or namespace not in raw_datasets:
                raise ValueError(f'Unknown sample ID: {sample_id}')
            dataset = raw_datasets[namespace]
            index = int(index)
            if int(dataset.targets[index]) != target:
                raise ValueError(f'Raw image/target alignment differs: {sample_id}')
            axis.imshow(dataset.data[index].cpu().numpy(), cmap='gray', vmin=0, vmax=255)
            axis.set_title(f'#{index}  true: {CLASS_NAMES[target]}\npred: {CLASS_NAMES[predicted_label]} ({confidence:.2f})', fontsize=8)
        figure.suptitle(f'{title} | {kind.replace("_", " ")} | n={len(selected)}', fontsize=12)
        figure.savefig(destination / f'gallery_{kind}.png', dpi=140)


def export_summary(destination, items, profile):
    import numpy as np
    from matplotlib.figure import Figure
    from matplotlib.backends.backend_agg import FigureCanvasAgg
    destination.mkdir()
    rows = []
    for item in items:
        root = profile / item['run'].name
        benchmark = read_json(root / 'benchmark' / 'benchmark.json')
        for split in ('validation', 'test'):
            record = read_json(root / split / 'metrics.json')
            rows.append({'model': item['model'], 'seed': item['seed'], 'run_id': item['run'].name,
                'eval_split': split, **record['metrics'],
                'parameter_count': benchmark['parameter_count'],
                'inference_ms_per_image': benchmark.get('amortized_ms_per_image'),
                'inference_device': benchmark.get('device'),
                'inference_status': benchmark['status']})
    aggregates = []
    for model in MODELS:
        for split in ('validation', 'test'):
            group = [r for r in rows if r['model'] == model and r['eval_split'] == split]
            if len(group) != 3 or {r['seed'] for r in group} != set(SEEDS):
                raise RuntimeError('Summary needs all three agreed seeds')
            counts = {r['parameter_count'] for r in group}
            if len(counts) != 1:
                raise RuntimeError('Parameter counts differ within one model')
            aggregate = {'model': model, 'eval_split': split, 'n': 3, 'parameter_count': counts.pop()}
            for metric in ('loss', 'accuracy', 'macro_f1'):
                values = [r[metric] for r in group]
                aggregate[metric + '_mean'] = mean(values)
                aggregate[metric + '_sample_sd'] = stdev(values)
            values = [r['inference_ms_per_image'] for r in group if r['inference_ms_per_image'] is not None]
            aggregate['inference_ms_per_image_mean'] = mean(values) if len(values) == 3 else None
            aggregates.append(aggregate)
    for name, data in [('per_run.csv', rows), ('mean_std.csv', aggregates)]:
        with (destination / name).open('w', newline='', encoding='utf-8') as stream:
            writer = csv.DictWriter(stream, fieldnames=list(data[0]))
            writer.writeheader()
            writer.writerows(data)
    save_json(destination / 'mean_std.json', aggregates)
    figure = Figure(figsize=(12, 8), constrained_layout=True)
    FigureCanvasAgg(figure)
    for axis, (split, metric) in zip(figure.subplots(2, 2).ravel(),
            [('validation', 'accuracy'), ('test', 'accuracy'), ('validation', 'macro_f1'), ('test', 'macro_f1')]):
        group = [r for r in aggregates if r['eval_split'] == split]
        values = [100*r[metric + '_mean'] for r in group]
        errors = [100*r[metric + '_sample_sd'] for r in group]
        axis.bar(range(5), values, yerr=errors, capsize=4, color='#2166ac')
        axis.set(xticks=range(5), xticklabels=MODELS, ylabel=f'{metric} (%)',
                 title=f'{split.capitalize()} | mean + sample SD | n=3', ylim=(0, 100))
        axis.tick_params(axis='x', labelrotation=20)
        for i, value in enumerate(values):
            axis.text(i, value + 1.2, f'{value:.2f}', ha='center', fontsize=9)
    figure.savefig(destination / 'comparison.png', dpi=160)
    total_time = sum(sum(float(r['epoch_seconds']) for r in csv.DictReader(
        (item['run'] / 'history.csv').read_text().splitlines())) for item in items)
    save_json(destination / 'coverage.json', {'runs': 15, 'evaluation_rows': len(rows),
        'sample_sd_ddof': 1, 'training_epoch_seconds_sum': total_time,
        'training_time_note': 'Train+validation loop sum from old histories, not total fit time or measured inference.',
        'test_policy': 'Saved best checkpoints only. No tuning or seed selection from test results.'})


def run_postprocess(artifact_root, checkout, output_root, device='auto', benchmark=True):
    import torch
    from dlbench.a1.analysis import analyze_run, analyze_predictions
    from dlbench.a1.benchmark import benchmark_inference
    from dlbench.a1.checkpoint import load_checkpoint
    from dlbench.a1.data.dataset import prepare_data, load_official_dataset
    from dlbench.a1.models.registry import build_model
    from dlbench.a1.trainer import evaluate_checkpoint
    from dlbench.common.artifacts import compute_data_provenance
    from dlbench.common.config import validate_config
    from dlbench.common.reproducibility import seed_everything, collect_environment
    os.chdir(checkout)
    ledger, items = inspect_inputs(artifact_root)
    actual_device = ('cuda' if torch.cuda.is_available() else 'cpu') if device == 'auto' else device
    if actual_device not in ('cpu', 'cuda') or (actual_device == 'cuda' and not torch.cuda.is_available()):
        raise RuntimeError('Requested device is unavailable')
    if actual_device == 'cpu':
        torch.set_num_threads(min(4, os.cpu_count() or 1))
    seed_everything(SEEDS[0])
    torch.set_float32_matmul_precision('highest')
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = False
    environment = collect_environment()
    profile_identity = {'version': VERSION, 'processor_sha256': globals().get('PROCESSOR_SHA256', VERSION),
        'device': actual_device, 'benchmark_requested': bool(benchmark),
        'cpu_threads': torch.get_num_threads(),
        'environment': {k: v for k, v in environment.items() if k != 'system_ram'}}
    profile = Path(output_root) / (actual_device + '-' + json_digest(profile_identity)[:12])
    profile.mkdir(parents=True, exist_ok=True)
    if not (profile / 'session.json').exists():
        save_json(profile / 'session.json', {'identity': profile_identity, 'environment': environment,
            'started_utc': datetime.now(timezone.utc).isoformat(), 'source_ledger': ledger['identity']})
    print('Evaluation device:', actual_device, '| output:', profile, flush=True)
    if actual_device == 'cpu':
        print('GPU unavailable/not selected: test+analysis use CPU; GPU inference timing will be SKIPPED.', flush=True)
    first = items[0]['config']
    split_file = Path(first['data']['split_file'])
    if not split_file.is_file() or sha256(split_file) != ledger['identity']['split_hash']:
        raise RuntimeError('Restore the exact ledger-pinned split manifest; do not regenerate it')
    prepared = prepare_data(deepcopy(first))
    for key in ('mean', 'std'):
        if any(abs(a-b) > 1e-6 for a,b in zip(prepared['measured_' + key], first['preprocessing'][key])):
            raise RuntimeError('Train-only normalization remeasurement differs')
    raw = {'official_train': load_official_dataset(first['data']['root'], train=True),
           'official_test': load_official_dataset(first['data']['root'], train=False)}
    for number, item in enumerate(items, 1):
        run, config = item['run'], deepcopy(item['config'])
        config['run']['device'] = actual_device  # Operational evaluation device only.
        validate_config(config, strict=True)
        if compute_data_provenance(config) != {k: ledger['identity'][k] for k in ('split_hash', 'statistics_hash')}:
            raise RuntimeError('Current data provenance differs')
        context = {'profile': profile_identity, 'run_id': run.name, 'inputs': item['hashes']}
        run_output = profile / run.name
        print(f'[{number}/15] {item["model"]} seed {item["seed"]}', flush=True)
        completed_stage(run_output / 'history', context,
            lambda path: analyze_run(run, output_dir=path))
        for split in ('validation', 'test'):
            def evaluate_stage(path, split=split):
                seed_everything(item['seed'])
                result = evaluate_checkpoint(config, run / 'best.pt', split=split)
                expected_count = config['data']['validation_size' if split == 'validation' else 'test_size']
                namespace = 'official_train' if split == 'validation' else 'official_test'
                if result.metrics.num_samples != expected_count or any(
                    not sid.startswith('fashion_mnist:' + namespace + ':') for sid in result.predictions.sample_ids):
                    raise RuntimeError('Evaluation sample count/namespace differs')
                metrics = asdict(result.metrics)
                replay = check_validation_replay(metrics, item['saved'], actual_device) if split == 'validation' else None
                analyze_predictions(result.predictions, path, split=split)
                save_json(path / 'metrics.json', {'run_id': run.name, 'model': item['model'],
                    'seed': item['seed'], 'eval_split': split, 'device': actual_device,
                    'checkpoint_sha256': item['hashes']['best.pt'], 'metrics': metrics,
                    'validation_replay': replay})
                prediction_extras(result.predictions, path, raw,
                    f'{item["model"]} | seed {item["seed"]} | {split}')
            completed_stage(run_output / split, context, evaluate_stage)
            record = read_json(run_output / split / 'metrics.json')['metrics']
            print(f'  {split}: accuracy={record["accuracy"]:.6f} macro-F1={record["macro_f1"]:.6f}', flush=True)
        def benchmark_stage(path):
            path.mkdir()
            payload = load_checkpoint(run / 'best.pt', map_location=actual_device)
            model = build_model(config['model']).to(actual_device)
            model.load_state_dict(payload['model_state_dict'])
            record = {'parameter_count': sum(p.numel() for p in model.parameters()),
                'trainable_parameters': sum(p.numel() for p in model.parameters() if p.requires_grad),
                'status': 'skipped_gpu_unavailable' if actual_device == 'cpu' else 'disabled',
                'requested_timing_config': config['timing'], 'environment': environment}
            if benchmark and actual_device == 'cuda':
                timing = deepcopy(config['timing'])
                if timing['device'] != 'cuda':
                    raise RuntimeError('Expected the saved CUDA timing protocol')
                shape = [timing['batch_size'], config['preprocessing']['channels'], *config['preprocessing']['image_size']]
                batch = torch.zeros(shape, dtype=torch.float32, device='cuda')
                record.update(benchmark_inference(model, batch, timing))
                record.update(status='completed', checkpoint_sha256=item['hashes']['best.pt'],
                    input_policy='Preallocated zeros; forward only; no loader/transfer/checkpoint I/O')
                del batch
            save_json(path / 'benchmark.json', record)
            del model, payload
            if actual_device == 'cuda':
                torch.cuda.empty_cache()
        completed_stage(run_output / 'benchmark', context, benchmark_stage)
    aggregate_identity = {'profile': profile_identity,
        'runs': {item['run'].name: item['hashes'] for item in items}}
    completed_stage(profile / 'summary', aggregate_identity,
        lambda path: export_summary(path, items, profile))
    archive_path = profile.with_suffix('.zip')
    if not archive_path.exists():
        temporary_zip = archive_path.with_suffix('.partial.zip')
        with zipfile.ZipFile(temporary_zip, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
            for path in sorted(profile.rglob('*')):
                if path.is_file() and not any(part.startswith('.') for part in path.relative_to(profile).parts):
                    archive.write(path, profile.name + '/' + path.relative_to(profile).as_posix())
        temporary_zip.replace(archive_path)
    print('\nDONE: 15/15 checkpoints, validation + test + analysis.', flush=True)
    print('Summary:', profile / 'summary' / 'mean_std.csv', flush=True)
    print('ZIP:', archive_path, flush=True)
    for row in read_json(profile / 'summary' / 'mean_std.json'):
        if row['eval_split'] == 'test':
            print(row['model'], f"test accuracy={100*row['accuracy_mean']:.3f} ± {100*row['accuracy_sample_sd']:.3f}%",
                  f"macro-F1={100*row['macro_f1_mean']:.3f} ± {100*row['macro_f1_sample_sd']:.3f}%")
    return profile, archive_path

PROCESSOR_SHA256 = '6668b16dee6aa449b31b908adb77fac6754bd2b962643492e4d4ce299373d86c'


## 3. Run all 15 checkpoints

Thứ tự mỗi run: kiểm tra history → replay validation → official test → ảnh/metrics → benchmark.
`SKIP verified` nghĩa là bước đã lưu đầy đủ và các file vẫn đúng hash, không phải bỏ sót run.
Folder `.partial-*` giữ đầu ra của bước bị ngắt; các bước hoàn tất được lưu với `_complete.json`.
Không xóa dữ liệu nguồn. CPU/GPU hoặc phiên bản môi trường khác nhau dùng profile output riêng.


In [ ]:
if source_git('rev-parse', 'HEAD') != pinned_commit or source_git('status', '--porcelain'):
    raise RuntimeError('Source changed after setup; stop before checkpoint evaluation.')
PROFILE_DIR, RESULT_ZIP = run_postprocess(
    artifact_root=ARTIFACT_ROOT,
    checkout=CHECKOUT,
    output_root=OUTPUT_ROOT,
    device=DEVICE,
    benchmark=RUN_GPU_BENCHMARK,
)


## 4. Review and download

`summary/mean_std.csv` chứa mean ± sample SD trên cả ba seed; `summary/per_run.csv` chứa từng checkpoint.
Mỗi run có `history/`, `validation/`, `test/`, `benchmark/` và các marker đã kiểm tra.
Nếu không có GPU, thông số model vẫn có nhưng cột inference để trống và status ghi rõ skipped.
Thông số `amortized_ms_per_image` tại batch 64 không phải latency của request batch-one.


In [ ]:
from IPython.display import Image, display

display(Image(filename=str(PROFILE_DIR / 'summary' / 'comparison.png')))
print('Drive output:', PROFILE_DIR)
print('ZIP:', RESULT_ZIP)
if AUTO_DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(RESULT_ZIP))
